# Prompt 5 - ML Infrastructure and Data Preparation

This notebook prepares the existing Prompt 3 feature table for future model training without fitting production models.


In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.eda import get_feature_columns, load_feature_table
from src.ml_pipeline import (
    MLConfig,
    default_ml_config,
    export_ml_outputs,
    prepare_ml_data,
    validate_no_participant_leakage,
)


In [2]:
features = load_feature_table('data/processed/features.csv')
features.shape


(258, 62)

In [3]:
predictor_columns = get_feature_columns(features)
len(predictor_columns), predictor_columns[:5]


(57,
 ['hip_flexion_right_mean',
  'hip_flexion_right_median',
  'hip_flexion_right_std',
  'hip_flexion_right_variance',
  'hip_flexion_right_maximum'])

In [4]:
config = default_ml_config()
config


MLConfig(random_seed=42, train_ratio=0.7, validation_ratio=0.15, test_ratio=0.15, missing_policy='drop_rows', scaler='standard', feature_selection='all', top_n_features=20, variance_threshold=0.0, correlation_threshold=0.95, cross_validation_folds=5, feature_ranking_path='reports/feature_ranking.csv')

In [5]:
result = prepare_ml_data(features, config)
result.split_summary


,split,participant_count,row_count,empty_trial_rows,complete_feature_rows,missing_feature_values
0,train,30,180,9,171,513
1,validation,6,36,0,36,0
2,test,7,42,0,42,0


In [6]:
result.leakage_summary


,check,overlap_count,overlapping_participants,passed
0,train_vs_validation,0,,True
1,train_vs_test,0,,True
2,validation_vs_test,0,,True


In [7]:
result.preprocessing_summary


,split,input_rows,output_rows,rows_removed_by_missing_policy,feature_count,missing_policy,scaler,fit_source
0,train,180,171,9,57,drop_rows,standard,train_only
1,validation,36,36,0,57,drop_rows,standard,train_only
2,test,42,42,0,57,drop_rows,standard,train_only


In [8]:
result.feature_selection_summary.head(10)


,feature,selected,strategy,reason
0,hip_flexion_right_mean,True,all,All Prompt 3 biomechanical features retained.
1,hip_flexion_right_median,True,all,All Prompt 3 biomechanical features retained.
2,hip_flexion_right_std,True,all,All Prompt 3 biomechanical features retained.
3,hip_flexion_right_variance,True,all,All Prompt 3 biomechanical features retained.
4,hip_flexion_right_maximum,True,all,All Prompt 3 biomechanical features retained.
5,hip_flexion_right_minimum,True,all,All Prompt 3 biomechanical features retained.
6,hip_flexion_right_rom,True,all,All Prompt 3 biomechanical features retained.
7,hip_flexion_right_time_to_peak,True,all,All Prompt 3 biomechanical features retained.
8,hip_flexion_left_mean,True,all,All Prompt 3 biomechanical features retained.
9,hip_flexion_left_median,True,all,All Prompt 3 biomechanical features retained.


In [9]:
result.cross_validation_summary


,fold,train_participant_count,validation_participant_count,train_row_count,validation_row_count,participant_overlap_count,validation_participants
0,1,34,9,204,54,0,"6,9,20,26,29,30,34,42,44"
1,2,34,9,204,54,0,"7,18,22,23,25,28,32,36,41"
2,3,34,9,204,54,0,"4,8,12,17,19,27,33,39,40"
3,4,35,8,210,48,0,"1,11,13,14,21,24,31,37"
4,5,35,8,210,48,0,"2,3,10,15,16,35,38,43"


In [10]:
paths = export_ml_outputs(result)
{name: str(path) for name, path in paths.items()}


{'split_summary': 'reports/split_summary.csv',
 'preprocessing_summary': 'reports/preprocessing_summary.csv',
 'feature_selection_summary': 'reports/feature_selection_summary.csv',
 'cross_validation_summary': 'reports/cross_validation_summary.csv',
 'leakage_summary': 'reports/leakage_summary.csv',
 'split_definitions': 'reports/ml_artifacts/split_definitions.csv',
 'preprocessing_artifact': 'reports/ml_artifacts/preprocessing_artifact.json',
 'feature_selection_artifact': 'reports/ml_artifacts/feature_selection_artifact.json',
 'config': 'configs/default_ml_config.yaml',
 'config_snapshot': 'reports/ml_artifacts/config_snapshot.yaml'}

In [11]:
top_n_config = MLConfig(feature_selection='top_n', top_n_features=10)
top_n_result = prepare_ml_data(features, top_n_config)
top_n_result.feature_selection_summary.head(12)


,feature,selected,strategy,reason
0,hip_flexion_right_variance,True,top_n,Selected top 10 features from Prompt 4 ranking.
1,knee_flexion_rom_symmetry_index,True,top_n,Selected top 10 features from Prompt 4 ranking.
2,hip_flexion_left_variance,True,top_n,Selected top 10 features from Prompt 4 ranking.
3,knee_flexion_rom_absolute_difference,True,top_n,Selected top 10 features from Prompt 4 ranking.
4,knee_flexion_rom_percent_difference,True,top_n,Selected top 10 features from Prompt 4 ranking.
5,hip_flexion_rom_symmetry_index,True,top_n,Selected top 10 features from Prompt 4 ranking.
6,hip_flexion_right_maximum,True,top_n,Selected top 10 features from Prompt 4 ranking.
7,hip_flexion_left_maximum,True,top_n,Selected top 10 features from Prompt 4 ranking.
8,ankle_angle_rom_symmetry_index,True,top_n,Selected top 10 features from Prompt 4 ranking.
9,ankle_angle_left_maximum,True,top_n,Selected top 10 features from Prompt 4 ranking.
